# Day 1 Morning - 最簡單的 AI Agent

## 1. 什麼是最簡單的 Agent？

一個 Agent 的核心就是三個東西：

```
Agent = LLM + Tools + Loop
```

最簡單的 Agent：給 LLM 幾個工具，讓它自己決定要用哪個，然後不斷循環直到任務完成。

```
使用者提出需求
    ↓
LLM 思考 → 決定呼叫哪個工具
    ↓
執行工具 → 回傳結果給 LLM
    ↓
LLM 繼續思考 → 決定下一步（呼叫工具 or 完成）
    ↓
重複直到 LLM 認為任務完成
```

**參考專案：** https://github.com/ywchiu/local_agentic_llm  
這個專案用最少的程式碼實作了一個 Agent，只給 LLM 4 個工具（write_file, read_file, run_command, list_files），就能讓它自主建構程式。

## 環境設定

In [1]:
%%capture
!pip install openai

In [10]:
import os
import json
import subprocess
import tempfile
from openai import OpenAI

# 設定 API Key
os.environ["OPENAI_API_KEY"] = ""

# 建立 OpenAI client
client = OpenAI()

# 建立工作目錄（用 temp 目錄避免污染環境）
WORKSPACE = tempfile.mkdtemp(prefix="agent_workspace_")
print(f"工作目錄: {WORKSPACE}")

工作目錄: /var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq


## 2. 定義工具 (Tools)

我們給 Agent 4 個最基本的工具，讓它可以操作檔案系統和執行指令：

In [11]:
# === 工具 1: 寫入檔案 ===
def write_file(path, content):
    """寫入檔案"""
    # 安全性：限制在工作目錄內
    full_path = os.path.join(WORKSPACE, path)
    os.makedirs(os.path.dirname(full_path) if os.path.dirname(full_path) else WORKSPACE, exist_ok=True)
    with open(full_path, 'w') as f:
        f.write(content)
    return f"File written: {full_path} ({len(content)} bytes)"

# === 工具 2: 讀取檔案 ===
def read_file(path):
    """讀取檔案"""
    full_path = os.path.join(WORKSPACE, path)
    with open(full_path, 'r') as f:
        return f.read()

# === 工具 3: 執行 shell 指令 ===
def run_command(command):
    """執行 shell 指令"""
    result = subprocess.run(
        command, shell=True, capture_output=True, text=True,
        timeout=30, cwd=WORKSPACE
    )
    return result.stdout + result.stderr

# === 工具 4: 列出目錄下的檔案 ===
def list_files(path='.'):
    """列出目錄下的檔案"""
    full_path = os.path.join(WORKSPACE, path)
    return '\n'.join(os.listdir(full_path))

# 測試一下
print(write_file("test.txt", "Hello Agent!"))
print(read_file("test.txt"))
print(list_files())

File written: /var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq/test.txt (12 bytes)
Hello Agent!
test.txt


## 3. 將工具轉換成 OpenAI Tool 格式

OpenAI 的 function calling 需要用 JSON Schema 描述每個工具的參數格式，LLM 才知道怎麼呼叫：

In [12]:
# OpenAI function calling 的工具定義格式
tools = [
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "寫入內容到指定檔案路徑",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "檔案路徑"},
                    "content": {"type": "string", "description": "檔案內容"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "讀取指定檔案的內容",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "檔案路徑"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "執行 shell 指令並回傳結果",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "要執行的 shell 指令"}
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "列出指定目錄下的所有檔案",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "目錄路徑，預設為當前目錄", "default": "."}
                },
                "required": []
            }
        }
    }
]

print(f"共定義了 {len(tools)} 個工具:")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

共定義了 4 個工具:
  - write_file: 寫入內容到指定檔案路徑
  - read_file: 讀取指定檔案的內容
  - run_command: 執行 shell 指令並回傳結果
  - list_files: 列出指定目錄下的所有檔案


## 4. 建立 Agent 迴圈

Agent 的核心就是一個迴圈：
1. 把訊息送給 LLM
2. 如果 LLM 回傳 tool_calls → 執行工具 → 把結果加回訊息 → 回到 1
3. 如果 LLM 沒有 tool_calls → 任務完成，跳出迴圈

In [14]:
# 工具名稱對應到實際的 Python 函數
tool_map = {
    "write_file": write_file,
    "read_file": read_file,
    "run_command": run_command,
    "list_files": list_files,
}

def execute_tool(name, args):
    """執行工具並回傳結果"""
    func = tool_map.get(name)
    if func is None:
        return f"Error: 未知的工具 '{name}'"
    try:
        return func(**args)
    except Exception as e:
        return f"Error: {str(e)}"


def run_agent(user_prompt, max_turns=10):
    """執行 Agent 迴圈"""
    print(f"{'='*60}")
    print(f"使用者需求: {user_prompt}")
    print(f"{'='*60}")

    messages = [
        {
            "role": "system",
            "content": "你是一個軟體工程師。用戶會請你建構東西。使用提供的工具來寫檔案和執行指令。完成後請總結你做了什麼。"
        },
        {"role": "user", "content": user_prompt}
    ]

    for turn in range(max_turns):
        print(f"\n--- 第 {turn + 1} 輪 ---")

        # 呼叫 LLM
        response = client.chat.completions.create(
            model="gpt-5.4-mini-2026-03-17",
            messages=messages,
            tools=tools,
            temperature=0
        )

        message = response.choices[0].message
        messages.append(message)

        # 如果沒有 tool_calls，表示 Agent 完成了
        if not message.tool_calls:
            print(f"\nAgent 完成！共 {turn + 1} 輪")
            print(f"最終回覆: {message.content}")
            break

        # 執行每個 tool call
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            # 印出工具呼叫（截斷太長的內容）
            args_display = {k: (v[:80] + '...' if isinstance(v, str) and len(v) > 80 else v) for k, v in args.items()}
            print(f"  [Tool] {name}({args_display})")

            result = execute_tool(name, args)
            print(f"  [Result] {str(result)[:200]}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

    return messages

## 5. 測試：讓 Agent 自己寫程式

我們給 Agent 一個任務，看它怎麼自主完成：

In [15]:
# 讓 Agent 寫一個 CSV 轉 JSON 的程式
messages = run_agent("寫一個 Python 程式，可以將 CSV 檔案轉換成 JSON 格式，要能處理不同的分隔符號和缺失值")

使用者需求: 寫一個 Python 程式，可以將 CSV 檔案轉換成 JSON 格式，要能處理不同的分隔符號和缺失值

--- 第 1 輪 ---
  [Tool] write_file({'path': 'csv_to_json.py', 'content': '#!/usr/bin/env python3\n"""Convert CSV files to JSON.\n\nSupports:\n- Custom delimit...'})
  [Result] File written: /var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq/csv_to_json.py (2786 bytes)

--- 第 2 輪 ---

Agent 完成！共 2 輪
最終回覆: 我已幫你寫好一個 Python 程式 `csv_to_json.py`，功能包括：

- 轉換 CSV 為 JSON
- 支援自訂分隔符號（例如 `,`、`;`、`\t`、`|`）
- 支援缺失值處理，可用 `--missing-value` 指定替代值
- 預設輸出為 JSON 陣列，每列 CSV 轉成一個物件

### 使用方式
```bash
python csv_to_json.py input.csv output.json
```

### 指定分隔符號
```bash
python csv_to_json.py input.csv output.json --delimiter ';'
python csv_to_json.py input.csv output.json --delimiter '\t'
```

### 指定缺失值替代
```bash
python csv_to_json.py input.csv output.json --missing-value null
```

### 程式特點
- 使用 `csv.DictReader` 讀取 CSV
- 將空值或缺失欄位轉成你指定的 `missing_value`
- 輸出 JSON 時保留中文，並使用縮排格式化

如果你要，我也可以再幫你加上：
- 自動偵測分隔符號
- 將數字/布林值自動轉型
- 支援輸出成 JSON L

In [16]:
# 看看 Agent 建立了哪些檔案
print("工作目錄中的檔案:")
for f in os.listdir(WORKSPACE):
    full = os.path.join(WORKSPACE, f)
    size = os.path.getsize(full)
    print(f"  {f} ({size} bytes)")

工作目錄中的檔案:
  test.txt (12 bytes)
  csv_to_json.py (2786 bytes)


In [23]:
#print(open(WORKSPACE + "/csv_to_json.py").read())

In [25]:
csv_content = """id,name,age,city,email
1,David,35,Taipei,david@example.com
2,Amy,29,Taichung,amy@example.com
3,John,42,Kaohsiung,john@example.com
4,Sophia,31,Tainan,sophia@example.com
"""

input_csv = WORKSPACE + "/input.csv"

with open(input_csv, "w", encoding="utf-8") as f:
    f.write(csv_content)

print(input_csv)

/var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq/input.csv


In [26]:
script_path = WORKSPACE + "/csv_to_json.py"
output_json = WORKSPACE + "/output.json"

!python {script_path} {input_csv} {output_json}

Converted /var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq/input.csv -> /var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq/output.json


In [27]:
!cat {output_json}

[
  {
    "id": "1",
    "name": "David",
    "age": "35",
    "city": "Taipei",
    "email": "david@example.com"
  },
  {
    "id": "2",
    "name": "Amy",
    "age": "29",
    "city": "Taichung",
    "email": "amy@example.com"
  },
  {
    "id": "3",
    "name": "John",
    "age": "42",
    "city": "Kaohsiung",
    "email": "john@example.com"
  },
  {
    "id": "4",
    "name": "Sophia",
    "age": "31",
    "city": "Tainan",
    "email": "sophia@example.com"
  }
]

## 6. 測試：讓 Agent 建立一個簡單的網頁

In [28]:
# 讓 Agent 建立 HP 筆電產品頁面
messages = run_agent("建立一個簡單的 HTML 頁面，顯示 HP 筆電產品列表，包含 EliteBook、Pavilion、Spectre 三個系列的基本規格")

使用者需求: 建立一個簡單的 HTML 頁面，顯示 HP 筆電產品列表，包含 EliteBook、Pavilion、Spectre 三個系列的基本規格

--- 第 1 輪 ---
  [Tool] write_file({'path': 'index.html', 'content': '<!DOCTYPE html>\n<html lang="zh-Hant">\n<head>\n  <meta charset="UTF-8" />\n  <meta ...'})
  [Result] File written: /var/folders/21/1tszsz8n1vdd79rbhmdnwwqc0000gn/T/agent_workspace_axr5eymq/index.html (2400 bytes)

--- 第 2 輪 ---

Agent 完成！共 2 輪
最終回覆: 已幫你建立一個簡單的 HTML 頁面 `index.html`，內容包含：

- HP 筆電產品列表標題
- 三個系列：
  - EliteBook
  - Pavilion
  - Spectre
- 每個系列的基本規格與特色
- 簡單的卡片式版面與基本 CSS 美化

如果你要，我也可以再幫你加上：
- 圖片
- 價格欄位
- 排序/篩選功能
- 響應式更完整的版面設計


In [ ]:
# 顯示 Agent 產生的 HTML
from IPython.display import HTML

# 找出 HTML 檔案
html_files = [f for f in os.listdir(WORKSPACE) if f.endswith('.html')]
if html_files:
    html_content = open(os.path.join(WORKSPACE, html_files[0])).read()
    print(f"顯示: {html_files[0]}")
    display(HTML(html_content))
else:
    print("沒有找到 HTML 檔案")

## 7. Agent 的核心概念解析

回顧我們剛剛建立的 Agent，它的運作機制其實非常簡單：

### Tool Use（工具使用）
- LLM 透過 **function calling** 決定要使用哪個工具
- 我們定義好工具的名稱、描述、參數格式（JSON Schema）
- LLM 會根據任務需求，自動產生正確的函數呼叫

### Agent Loop（代理迴圈）
- 迴圈持續執行，直到 LLM 認為任務完成（不再呼叫工具）
- 每次迴圈：LLM 思考 → 呼叫工具 → 取得結果 → 繼續思考
- 設定 `max_turns` 防止無限迴圈

### Autonomy（自主性）
- LLM **自主決定**下一步行動：要用什麼工具、傳什麼參數
- 我們不需要寫 if-else 判斷邏輯，LLM 自己規劃執行步驟
- 這就是 Agent 和傳統程式最大的差別

### 這個簡單 Agent 的架構

```
┌─────────────────────────────────────┐
│           Agent Loop                │
│                                     │
│  User Prompt                        │
│       ↓                             │
│  ┌─────────┐    ┌──────────────┐    │
│  │   LLM   │───→│  Tool Calls  │    │
│  │(gpt-5.4 │    │              │    │
│  └────↑────┘    └──────┬───────┘    │
│       │                ↓            │
│       │         ┌──────────────┐    │
│       └─────────│ Tool Results │    │
│                 └──────────────┘    │
│                                     │
│  Tools: write_file, read_file,      │
│         run_command, list_files     │
└─────────────────────────────────────┘
```

---

## 8. Harness Engineering — 韁繩工程

> **Agent = 模型 + Harness**
>
> 模型負責「聰明」，Harness 負責「穩定交付」

我們剛才的 Agent 展示了「模型很聰明」的部分。但在真實場景中，光靠模型聰明是不夠的——你需要一套**系統駕馭機制**。

---

### AI 工程三階段演進

以「派新人拜訪客戶」為例：

| 階段 | 做法 | 核心問題 | 關鍵字 |
|------|------|---------|--------|
| **Stage 1** Prompt Engineering | 告訴新人流程：寒暄 → 介紹方案 → 詢問需求 → 確認下一步 | 模型聽不聽得懂 | 語言設計 |
| **Stage 2** Context Engineering | 準備好客戶背景、溝通記錄、產品報價、競品情況、會議目標 | 模型有沒有正確資訊 | 資訊供給（RAG）|
| **Stage 3** Harness Engineering | 帶 Checklist、關鍵節點即時匯報、會後交錄音紀要、偏差馬上糾正、按標準驗收 | 執行過程中能不能持續做對 | **系統駕馭** |

---

### Harness 六層架構

```
給對資料 → 配好工具 → 排好路線 → 記住進度 → 獨立驗收 → 設好護欄
```

| 層 | 名稱 | 類比 | 技術對應 |
|----|------|------|---------|
| ① | 資訊邊界層 | 只給他該知道的 | RAG、Context Window 管理 |
| ② | 工具系統層 | 給他對的武器 | Tool Calling、API 整合 |
| ③ | 執行編排層 | 幫他排好路線 | LangGraph State Machine |
| ④ | 記憶與狀態層 | 讓他記得做到哪了 | Memory、Checkpoint |
| ⑤ | 評估與觀測層 | 派人去抽查 | Grader、Evaluator、Tracing |
| ⑥ | 約束、校驗與恢復層 | 畫紅線 + 裝安全網 | Guardrails、Fallback、Retry |

---

### 頂尖公司怎麼做

**Anthropic — Context Reflection**
- 對話太長 → Agent「焦慮」、遺漏細節
- 解法：不壓縮，直接**重啟乾淨 Agent + 交接狀態**
- 角色拆分：Planner → Generator → Evaluator → 形成「生成 — 檢查 — 修復」循環

**OpenAI — 三個關鍵設計**
- **環境設計者**：人類不再寫程式碼 → 拆解目標、提供環境、建立反饋鏈路
- **目錄式引導**：目錄索引頁，按需查詢子文件（vs. 把整份文件塞給 Agent）
- **自動化治理**：資深工程師經驗寫入系統 → Agent 出錯自動攔截 + 反饋修復

---

> **模型決定產品上限，Harness 決定產品能否落地**

In [30]:
# 清理工作目錄
import shutil
shutil.rmtree(WORKSPACE, ignore_errors=True)
print("工作目錄已清理")

工作目錄已清理


## 8. 從簡單 Agent 到 LangGraph

我們剛剛建立的 Agent 雖然能動，但有幾個明顯的限制：

| 限制 | 說明 | LangGraph 解法 |
|------|------|----------------|
| **沒有狀態管理** | 只有 messages 列表，無法追蹤複雜狀態 | `StateGraph` 管理結構化狀態 |
| **沒有分支邏輯** | 只能線性執行，無法根據條件走不同路徑 | `add_conditional_edges` 條件路由 |
| **沒有記憶** | 重新執行就什麼都忘了 | `Checkpointer` 持久化記憶 |
| **沒有人類介入** | Agent 自己跑到底，無法中途確認 | `interrupt` 人機協作 |
| **沒有錯誤處理** | 工具失敗就 GG | 節點級別的錯誤處理和重試 |
| **沒有可觀測性** | 看不到 Agent 內部在幹嘛 | LangSmith 追蹤和除錯 |
